# 02 - Feature Engineering

基于采样后的交互数据，构建三类特征：
- **用户侧特征**：活跃度、关注/粉丝数、注册天数等
- **视频侧特征**：时长、类型、每日播放/点赞统计等
- **统计交叉特征**：用户历史行为均值、视频全局热度等

> **注意**：`item_daily_features` 是按日统计的特征，join 时需对齐交互日期，避免数据泄露。

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "./data/KuaiRec/data"
PROCESSED_DIR = "./data/processed"

## 1. 加载数据

In [ ]:
# 加载采样后的交互数据
df = pd.read_parquet(f"{PROCESSED_DIR}/interactions_sampled.parquet")
print(f"Interactions shape : {df.shape}")
print(f"Positive rate      : {df['label'].mean():.2%}")
df.head(3)

In [ ]:
# 加载用户特征
user_feat = pd.read_csv(f"{DATA_DIR}/user_features.csv")
print(f"User features shape: {user_feat.shape}")
user_feat.head(3)

In [ ]:
# 加载视频日统计特征（选取核心字段，避免维度爆炸）
ITEM_COLS = [
    'video_id', 'date', 'author_id', 'video_type', 'upload_dt', 'upload_type',
    'video_duration', 'video_width', 'video_height', 'music_id',
    'video_tag_id', 'video_tag_name',
    'show_cnt', 'play_cnt', 'play_user_num',
    'complete_play_cnt', 'valid_play_cnt',
    'like_cnt', 'comment_cnt', 'follow_cnt', 'share_cnt', 'collect_cnt',
    'play_progress'
]
item_feat = pd.read_csv(f"{DATA_DIR}/item_daily_features.csv", usecols=ITEM_COLS)
print(f"Item daily features shape: {item_feat.shape}")
item_feat.head(3)

In [ ]:
# 加载视频类别
item_cat = pd.read_csv(f"{DATA_DIR}/item_categories.csv")
print(f"Item categories shape: {item_cat.shape}")
item_cat.head(3)

## 2. 日期对齐处理

`small_matrix` 中 `date` 格式为 `20200705.0`（float），
`item_daily_features` 中 `date` 格式为 `20200705`（int）。
需统一为字符串 `"20200705"` 后进行 join。

In [ ]:
# 先丢弃 date 为 NaN 的行（无法对齐日统计特征）
df = df.dropna(subset=['date'])
df['date'] = df['date'].astype(float).astype(int).astype(str)
item_feat['date'] = item_feat['date'].astype(str)

print("Interaction date sample:", df['date'].iloc[0])
print("Item feat date sample  :", item_feat['date'].iloc[0])
print(f"Rows after dropna: {len(df):,}")

## 3. Join 用户特征

In [ ]:
df = df.merge(user_feat, on='user_id', how='left')
print(f"After user join: {df.shape}")
print(f"User feature null rate: {df['user_active_degree'].isna().mean():.2%}")

## 4. Join 视频日统计特征（日期对齐）

In [ ]:
df = df.merge(item_feat, on=['video_id', 'date'], how='left')
print(f"After item join: {df.shape}")
print(f"Item feature null rate (play_cnt): {df['play_cnt'].isna().mean():.2%}")

## 5. Join 视频类别特征

In [ ]:
# feat 列格式为字符串 "[8]"，解析为列表后取主类别
item_cat['feat_parsed'] = item_cat['feat'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) and x != '[]' else []
)
item_cat['main_category'] = item_cat['feat_parsed'].apply(
    lambda x: x[0] if len(x) > 0 else -1
)

df = df.merge(item_cat[['video_id', 'main_category']], on='video_id', how='left')
print(f"After category join: {df.shape}")

## 6. 构造统计交叉特征

> 注意：此处统计特征基于全量采样数据计算，包含了测试集信息，严格意义上存在轻微的数据泄露。
> 生产环境中，应仅在训练集上计算后再 join 到验证集/测试集。

In [ ]:
# 用户历史行为统计（基于 watch_ratio 连续值，非 label）
user_stats = df.groupby('user_id').agg(
    user_avg_watch_ratio=('watch_ratio', 'mean'),
    user_total_interactions=('video_id', 'count'),
    user_watch_ratio_std=('watch_ratio', 'std'),
).reset_index()

df = df.merge(user_stats, on='user_id', how='left')
print("User stats features added.")

In [ ]:
# 视频全局热度统计
video_stats = df.groupby('video_id').agg(
    video_avg_watch_ratio=('watch_ratio', 'mean'),
    video_interaction_cnt=('user_id', 'count'),
).reset_index()

df = df.merge(video_stats, on='video_id', how='left')
print("Video stats features added.")

In [ ]:
# 视频完播率（由日统计特征衍生）
df['complete_play_rate'] = df['complete_play_cnt'] / (df['play_cnt'] + 1)
df['valid_play_rate'] = df['valid_play_cnt'] / (df['play_cnt'] + 1)
df['like_play_rate'] = df['like_cnt'] / (df['play_cnt'] + 1)

## 7. 类别特征编码

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['user_active_degree', 'video_type', 'upload_type']
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        print(f"Encoded {col}: {df[col].nunique()} categories")

## 8. 特征选择与缺失值处理

In [ ]:
# 丢弃非特征列（ID、时间戳、原始标签来源）
DROP_COLS = [
    'user_id', 'video_id', 'play_duration', 'video_duration_x',
    'time', 'date', 'timestamp', 'watch_ratio',
    'upload_dt', 'video_tag_name',  # 高基数文本列
]

feature_cols = [c for c in df.columns if c not in DROP_COLS + ['label']]
print(f"Total features: {len(feature_cols)}")
print(feature_cols)

In [ ]:
df_features = df[feature_cols + ['label']].copy()

# 缺失值情况
null_rate = df_features.isna().mean().sort_values(ascending=False)
print("Top missing value columns:")
print(null_rate[null_rate > 0].head(20))

In [ ]:
# 数值特征用中位数填充，类别特征用 -1 填充
num_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != 'label']

df_features[num_cols] = df_features[num_cols].fillna(df_features[num_cols].median())

print(f"Remaining nulls: {df_features.isna().sum().sum()}")
print(f"Final dataset shape: {df_features.shape}")

## 9. 特征分布概览

In [ ]:
# 正负样本在关键特征上的分布差异
key_features = ['user_avg_watch_ratio', 'video_avg_watch_ratio', 'complete_play_rate', 'register_days']

fig, axes = plt.subplots(1, len(key_features), figsize=(16, 4))
for i, feat in enumerate(key_features):
    if feat not in df_features.columns:
        continue
    for label_val, color, name in [(0, 'steelblue', 'Negative'), (1, 'coral', 'Positive')]:
        subset = df_features[df_features['label'] == label_val][feat]
        axes[i].hist(subset.clip(subset.quantile(0.01), subset.quantile(0.99)),
                     bins=40, alpha=0.6, color=color, label=name, density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distribution by Label', y=1.02)
plt.tight_layout()
plt.show()

## 10. 保存特征

In [ ]:
output_path = f"{PROCESSED_DIR}/features.parquet"
df_features.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {df_features.shape}")
print(f"Positive rate: {df_features['label'].mean():.2%}")